# Prerequisites
본 `ipynb` 은 `Python=3.12` 에서 작성하였습니다. Package dependency 를 해결하기 위해 아래 cell 을 실행해주세요.

## Install Python packages

In [ ]:
%pip -q install -U agent-framework==1.0.0b260130 patch-ng unidiff

## Load environment variables from a .env file
secret 노출을 피하고 notebook 들간의 일관된 환경변수를 설정하기 위해 `dotenv` 을 이용한다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_MS_FOUNDRY_PROJECT_ENDPOINT = os.getenv("AZURE_MS_FOUNDRY_PROJECT_ENDPOINT")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")

# Simple Code Assistant
일반적인 coding agent 에서 사용하는 파일 시스템 도구들을 만들어보자. 파일 시스템을 조작할 수 있는 도구들을 정의한다:
- `read`: 파일 내용 읽기
- `write`: 파일에 내용 쓰기 (덮어쓰기)
- `ls`: 디렉토리 내용 나열
- `diff`: Unified diff 형식 편집

In [ ]:
from pathlib import Path
from typing import Annotated
from pydantic import Field
import patch_ng
from agent_framework import tool

@tool(name="read", description="파일 내용 읽기")
def read(file_path: Annotated[str, Field(description="파일 경로")]) -> str:
    try:
        path = Path(file_path).resolve()
        if not path.exists():
            return f"❌ 오류: 파일 '{file_path}'이(가) 존재하지 않습니다."
        if not path.is_file():
            return f"❌ 오류: '{file_path}'은(는) 파일이 아닙니다."
        
        content = path.read_text(encoding="utf-8")
        return f"✅ 파일 '{file_path}' 내용:\n{content}"
    except Exception as e:
        return f"❌ 파일 읽기 오류: {str(e)}"

@tool(name="write", description="파일에 내용 쓰기")
def write(
    file_path: Annotated[str, Field(description="파일 경로")],
    content: Annotated[str, Field(description="쓸 내용")]
) -> str:
    try:
        path = Path(file_path).resolve()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(content, encoding="utf-8")
        return f"✅ 파일 '{file_path}'에 성공적으로 내용을 저장했습니다. ({len(content)} 글자)"
    except Exception as e:
        return f"❌ 파일 쓰기 오류: {str(e)}"

@tool(name="diff", description="unified diff 형식으로 파일 편집")
def diff(
    file_path: Annotated[str, Field(description="파일 경로")],
    patch: Annotated[str, Field(description="unified diff 텍스트 (라인 번호를 포함)")],
) -> str:
    path = Path(file_path).resolve()
    if not path.exists():
        return f"❌ 파일이 존재하지 않습니다: {file_path}"
    if not path.is_file():
        return f"❌ 파일이 아닙니다: {file_path}"

    # patch 적용
    try:
        pset = patch_ng.fromstring(patch.encode("utf-8"))
        patched = pset.apply(root=str(Path()), strip=1)
    except Exception as e:
        return f"❌ 패치 적용 실패: {e}"

    return "패치 적용 완료"

@tool(name="ls", description="디렉토리 내용 나열")
def ls(
    dir_path: Annotated[str, Field(description="디렉토리 경로 (비어있으면 루트)")] = ""
) -> str:
    path = Path(dir_path).resolve()
    if not path.exists():
        return f"❌ 오류: 디렉토리 '{dir_path}'이(가) 존재하지 않습니다."
    if not path.is_dir():
        return f"❌ 오류: '{dir_path}'은(는) 디렉토리가 아닙니다."

    try:
        items = []
        for item in sorted(path.iterdir()):
            item_type = "📁" if item.is_dir() else "📄"
            size = item.stat().st_size if item.is_file() else "-"
            items.append(f"  {item_type} {item.name} ({size} bytes)" if item.is_file() else f"  {item_type} {item.name}/")
        
        result = f"✅ 디렉토리 '{dir_path or '.'}' 내용:\n" + "\n".join(items) if items else f"디렉토리 '{dir_path or '.'}' 가 비어있습니다."
        return result
    except Exception as e:
        return f"❌ 디렉토리 나열 오류: {str(e)}"

# 도구 목록
file_tools = [read, write, diff, ls]
print("✅ Tools 정의 완료:", [t.name for t in file_tools])

위에서 정의한 파일 시스템 도구들을 사용하는 에이전트를 만들어보자.

In [ ]:
from agent_framework import ChatAgent
from agent_framework.openai import OpenAIResponsesClient

# Code Assistant 생성
code_assistant = ChatAgent(
    chat_client=OpenAIResponsesClient(
        base_url=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_API_KEY,
        model_id=AZURE_OPENAI_CHAT_DEPLOYMENT,
    ),
    name="CodeAssistant",
    instructions="""너는 파일 시스템을 관리하는 도우미야. 
사용자의 요청에 따라 파일을 읽고, 쓰고, 삭제하고, 디렉토리를 탐색할 수 있어.
항상 작업 결과를 명확하게 알려줘.
모든 파일 경로는 workspace 디렉토리 기준 상대 경로야.""",
    tools=file_tools,
)

Agent 의 응답을 출력하는 함수를 정의합니다.

In [ ]:
from agent_framework import Role

def print_result(result):
    """에이전트 실행 결과를 포맷팅하여 출력"""
    
    # 도구 호출 정보 출력
    print("\n======= 🤖 Agent Messages =======")
    for idx,msg in enumerate(result.messages):
        if msg.role == Role.ASSISTANT:
            for content in msg.contents:
                if content.type == "text":
                    print(f"[{idx}] 💬 {content.text}")
                elif content.type == "function_call":
                    print(f"[{idx}] 🔧 {content.type} | {content.name} | {content.raw_representation.arguments}")
                else:
                    print(f"[{idx}] not supported content type: {content.type}")
        elif msg.role == Role.TOOL:
            for content in msg.contents:
                print(f"[{idx}] 📤 {content.result}")
        else:
            print(f"[{idx}] not supported role: {msg.role}")

### 테스트: Assistant 의 Tools 동작 확인

In [ ]:
# 테스트: 에이전트 생성 확인 및 기본 실행
thread = code_assistant.get_new_thread()
result = await code_assistant.run(
    "현재 작업 디렉토리(.)의 내용을 나열해줘.",
    thread=thread
)
print_result(result)

In [ ]:
thread = code_assistant.get_new_thread()

# 파일 생성 테스트
result = await code_assistant.run(
    "notes 폴더에 hello.txt 파일을 만들고 '안녕하세요! 이것은 테스트 파일입니다.' 라고 써줘.",
    thread=thread
)

print_result(result)

In [ ]:
# 테스트: 파일 읽기 테스트
result = await code_assistant.run(
    "notes 폴더에 hello.txt 파일의 내용을 읽어줘.",
    thread=thread,
)

print_result(result)

In [ ]:
# 테스트: 복합 작업, notes 폴더에 메모 파일 생성 후 목록 확인
result = await code_assistant.run(
    """다음 작업을 수행해줘:
    1. notes 폴더에 meeting.txt 파일을 만들고 '회의 내용: AI 프로젝트 진행 상황 공유' 라고 써줘
    2. notes 폴더에 todo.txt 파일을 만들고 '할 일: 보고서 작성, 코드 리뷰' 라고 써줘
    3. notes 폴더의 파일 목록을 보여줘""",
    thread=thread,
)

print_result(result)

In [ ]:
# 테스트: 파일 정밀 편집 테스트
result = await code_assistant.run(
    """diff 도구를 이용해서 notes 폴더에 todo.txt 파일에서 '보고서 작성'을 '보고서 작성 완료'로 수정한 다음, 파일 전체 내용을 보여줘.""",
    thread=thread
)

print_result(result)

## Human-in-the-Loop 적용
파일 쓰기나 삭제와 같은 위험한 작업은 사용자 승인을 필요로 하도록 설정할 수 있다.

In [ ]:
from agent_framework import ChatMessage, Role

# 읽기 작업은 승인 불필요, 쓰기 작업은 승인 필요
@tool(name="safe_read", description="파일 내용 읽기", approval_mode="never_require")
def safe_read(
    file_path: Annotated[str, Field(description="파일 경로")]
) -> str:
    return read(file_path)

@tool(name="safe_write", description="파일에 내용 쓰기", approval_mode="always_require")
def safe_write(
    file_path: Annotated[str, Field(description="파일 경로")],
    content: Annotated[str, Field(description="쓸 내용")]
) -> str:
    return write(file_path, content)

@tool(name="safe_diff", description="Unified diff 형식으로 파일 편집", approval_mode="always_require")
def safe_diff(
    path: Annotated[str, Field(description="파일 경로")],
    patch: Annotated[str, Field(description="git 스타일 unified diff 텍스트")],
) -> str:
    return diff(path, patch)

safe_file_tools = [safe_read, safe_write, safe_diff, ls]
print("✅ Safe File Tools 정의 완료 (HITL 적용):", [t.name for t in safe_file_tools])

In [ ]:
# HITL이 적용된 파일 시스템 에이전트
safe_code_assistant = ChatAgent(
    chat_client=OpenAIResponsesClient(
        base_url=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_API_KEY,
        model_id=AZURE_OPENAI_CHAT_DEPLOYMENT,
    ),
    name="SafeCodeAssistant",
    instructions="""너는 안전한 파일 시스템 관리 도우미야.""",
    tools=safe_file_tools,
)

## 테스트: HITL 동작 확인

In [ ]:
# 테스트: 복합 테스트
thread = safe_code_assistant.get_new_thread()
result = await safe_code_assistant.run(
    "notes 폴더에 important.txt 파일을 만들고 '중요한 데이터입니다.' 라고 써줘.",
    thread=thread
)

# 승인 요청 처리
if result.user_input_requests:
    for user_input_needed in result.user_input_requests:
        print(f"🔐 승인 요청:")
        print(f"   함수: {user_input_needed.function_call.name}")
        print(f"   인수: {user_input_needed.function_call.arguments}")
        approval = input("승인하시겠습니까? (y/n): ")
        approval = True if approval.lower() == "y" else False
        
        approval_message = ChatMessage(
            role=Role.USER, 
            contents=[user_input_needed.to_function_approval_response(approval)]
        )
        
        final_result = await safe_code_assistant.run(approval_message, thread=thread)
        print_result(final_result)
else:
    print_result(result)

# Cleanup
테스트로 생성된 workspace 폴더를 정리한다.

In [ ]:
import shutil

# workspace 폴더 삭제 (주의: 모든 테스트 파일이 삭제됩니다)
workspace_dir = Path("notes")
if workspace_dir.exists():
    shutil.rmtree(workspace_dir)
    print(f"'{workspace_dir}' 폴더가 삭제되었습니다.")
else:
    print(f"'{workspace_dir}' 폴더가 이미 존재하지 않습니다.")